In [ ]:
import pandas as pd
import csv
import numpy as np

from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import networkx as nx
from Bio import SeqIO

from matplotlib import pyplot as plt
from matplotlib import cm, colors
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import colorcet
import colorsys
from matplotlib.lines import Line2D
from adjustText import adjust_text
import textalloc as ta

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.godag_plot import plot_gos, plot_results, plot_goid2goobj

import re

import mathieu as mh
from progressbar import ProgressBar

In [ ]:
base_dir = '/home/mathieu/postdoc_heasley/rna_seq/'
#fig_path = f'{base_dir}fig/'
fig_path = '/home/mathieu/postdoc_heasley/publications/subtelomere_paper/fig/'

strains_info = pd.read_csv(f'{base_dir}script/strains_info.csv').set_index('sample')
strains_info = strains_info.loc[strains_info['project']=='MH']
collection = pd.read_excel('/home/mathieu/postdoc_heasley/collection/mccusker_collection_wild_annot.xlsx')
collection.index = collection['ID'].values

In [ ]:
strain_order = [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]
strain_order = collection.loc[strain_order].sort_values(by='Y\' element').index
strain_color = dict(zip(strain_order, colorcet.glasbey_category10))
strain_order_rank = dict(zip(strain_order, range(10)))
#strain_yprime_color = dict(zip(strain_order, [cm.viridis(i) for i in collection.loc[strain_order, 'Y\' element']/150]))
strain_yprime_color = dict(zip(strain_order, [cm.viridis(i) for i in np.linspace(0,1,10)]))
hu_high_color = dict(zip([0]+[2**i for i in range(4,11)], [cm.magma(i) for i in np.linspace(0,1,8)]))

In [ ]:
chr_list = pd.read_csv('../../short_read_project/data/ref/S288C.chr_list.tsv', sep='\t')
chr_ctg_to_alias = chr_list.set_index('contig_name')['alias'].to_dict()
chr_alias_to_ctg = {j:i for (i,j) in chr_ctg_to_alias.items()}

with open('../../short_read_project/data/ref/S288C.fasta') as fi:
    ref_genome = [seq for seq in SeqIO.parse(fi, 'fasta')]

# Genome display variables
chrom_order = list(chr_list['contig_name'])
ref_chr_len = [len(seq.seq) for seq in ref_genome]
ref_chr_len_dict = dict(zip(chrom_order, ref_chr_len))

tig_off = pd.Series(ref_chr_len_dict, name='len')
tig_off = pd.concat([tig_off, tig_off.cumsum().rename('cumul_end')], axis=1)
tig_off['cumul_start'] = np.concatenate([[0], tig_off['len'].iloc[:-1].cumsum().values])

# Generate annotation tables for featurecounts and deseq

In [ ]:
gff = pd.read_csv(f'{base_dir}data/ref/S288C.gff', sep='\t', comment='#', header=None)
gff_annot_exploded = pd.concat([gff.iloc[:, :8], gff[8].apply(lambda x: pd.Series(dict([i.split('=') for i in x.split(';')])))], axis=1)

gff_annot_exploded['bed_name'] = ''
for at, df in gff_annot_exploded.groupby(2):
    df = df.iloc[:, 8:].dropna(axis=1, how='any')
    gff_annot_exploded.loc[df.index, 'bed_name'] = df.apply(lambda x: ' '.join(x.astype(str))\
                                                            .replace('%20', ' ').replace('%3B', ';').replace('%2C', ',')\
                                                            .replace('%28', '(').replace('%29', ')').strip(), axis=1)

In [ ]:
gff_featurecounts = gff_annot_exploded.loc[gff_annot_exploded[2]=='gene'].reset_index(drop=True).copy()
# manually add missing IDs
for c in ['curie', 'dbxref']:
    gff_featurecounts.loc[gff_featurecounts['ID']=='YGR200C', c] = 'SGD:S000003432'
    gff_featurecounts.loc[gff_featurecounts['ID']=='YLR132C', c] = 'SGD:S000004122'

for strand, df in gff_featurecounts.groupby(6):
    
    if strand == '+':
        pos = df[4]
        gff_featurecounts.loc[df.index, 3] = pos-250
        gff_featurecounts.loc[df.index, 4] = pos+350
        
    elif strand == '-':
        pos = df[3]
        gff_featurecounts.loc[df.index, 3] = pos-350
        gff_featurecounts.loc[df.index, 4] = pos+250
gff_featurecounts[3] = np.where(gff_featurecounts[3]<1, 1, gff_featurecounts[3])
gff_featurecounts[8] = gff_featurecounts['ID'].apply(lambda x: f'Name \"{x}\"')
# create gtf for export
gtf_featurecounts = gff_featurecounts[range(9)].astype({3:int, 4:int})
gtf_featurecounts.loc[gtf_featurecounts.shape[0]] = pd.Series(['Y_prime_element.cons_trim', 'manual', 'gene', 3750, 4350, '.', '+', '.', 'Name \"Y_prime_short\"'])
gtf_featurecounts.loc[gtf_featurecounts.shape[0]] = pd.Series(['Y_prime_element.cons_trim', 'manual', 'gene', 5650, 6250, '.', '+', '.', 'Name \"Y_prime_long\"'])

#gtf_featurecounts.to_csv(f'{base_dir}data/ref/featurecounts.gtf', sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE)

In [ ]:
# Export annotations in BED format; for genome browser only.

bed = gff_annot_exploded.loc[gff_annot_exploded[2]=='gene'].reset_index(drop=True).copy()
bed[3] -= 1
bed['chrom'] = bed[0]
bed['strand'] = bed[6]
bed['name'] = bed['ID']
for strand, df in bed.groupby('strand'):
    
    if strand == '+':
        bed.loc[df.index, 'start'] = df[4]-250
        bed.loc[df.index, 'end'] = df[4]+350
        bed.loc[df.index, 'itemRGB'] = '255,0,0'
        
    elif strand == '-':
        bed.loc[df.index, 'start'] = df[3]-350
        bed.loc[df.index, 'end'] = df[3]+250
        bed.loc[df.index, 'itemRGB'] = '0,0,255'
    
bed['score'] = '.'
bed['thickStart'] = bed['start']
bed['thickEnd'] = bed['end']
#bed[['chrom','start','end','ID','score','strand','thickStart','thickEnd','itemRGB']].astype({'start':int, 'end':int, 'thickStart':int, 'thickEnd':int}).\
#to_csv(f'{base_dir}data/ref/featurecounts.bed', sep='\t', index=False, header=False)

In [ ]:
# dictionary to translate systematic names into SGD ID
sysname_to_sgdid = gff_featurecounts.set_index('ID')['curie'].apply(lambda x: x.split(':')[1]).to_dict()
sgdid_to_sysname = {j:i for i,j in sysname_to_sgdid.items()}

sysname_to_name = pd.read_csv(f'{base_dir}script/sysname_to_name.csv').set_index('sys_name', drop=False)

# Regulation

In [ ]:
regulation = pd.read_csv(f'{base_dir}data/ref/alliancemine_results_2026-04-06T14-58-57.tsv', sep='\t')

# Build graph of regulatory interactions
Regulation = nx.DiGraph()

Regulation.add_nodes_from(gff_featurecounts['ID'].values)

for (gene, regulator), df in regulation.groupby(['Gene.secondaryIdentifier', 'Gene.regulatoryRegions.regulator.secondaryIdentifier']):
    Regulation.add_edge(gene, regulator)

# extract the list of targets
targets = regulation.groupby('Gene.regulatoryRegions.regulator.secondaryIdentifier')\
.apply(lambda x: x['Gene.secondaryIdentifier'].unique(), include_groups=False)

# Read counts and RPM

In [ ]:
counts = pd.read_csv(f'{base_dir}tables/counts.csv', index_col=0)
# transform into rpm long-format table
RPM = counts/counts.sum()*1e6
#RPM.loc['yprime'] = RPM.loc['Y_prime_long'] + RPM.loc['Y_prime_short']
RPM = RPM.reset_index(names='gene').melt(id_vars='gene', var_name='lib', value_name='rpm')
RPM['sample'] = RPM['lib'].apply(lambda x: x.split('.')[0])
RPM = RPM.merge(strains_info.reset_index())

In [ ]:
rpm = RPM.loc[RPM['IC']==0].groupby(['background','gene'])['rpm'].mean()

legend_elms = []

for s in ['YJM967', 'YJM964', 'YJM947', 'YJM955']:
    dat = rpm.loc[s].copy()
    dat.drop(['Y_prime_long', 'Y_prime_short'], inplace=True)
    dat = dat[dat>0].sort_values()
    print(s, (dat.loc['yprime']/1e6)*100)

# Parse the DESEQ results

In [ ]:
terms = ['complete.IC', 'complete.y_prime_kb', 'complete.IC.y_prime_kb', 'single_yprime.y_prime_kb', 'single_IC.IC']

In [ ]:
DESEQ = {}
for t in terms:
    deseq = pd.read_csv(f'{base_dir}tables/deseq.{t}.csv').rename({'Unnamed: 0':'gene'}, axis=1)
    deseq = deseq.loc[~deseq['log2FoldChange'].isna()]
    deseq['log_padj'] = np.where(deseq['padj'].isna(), np.nan, np.log10(deseq['padj'])*(-1))
    deseq['term'] = t
    DESEQ[t] = deseq

# GO enrichment

In [ ]:
# Import ontology and gaf
basic_obo = GODag(f'{base_dir}goatools/go-basic.obo')
sgd_gaf = GafReader(f'{base_dir}goatools/sgd.gaf')

In [ ]:
# define exclusion of GO terms for output, based on generality (number of children terms)
go_exclude = [go for (go, go_term) in basic_obo.items() if go_term.level == 0]
go_name = {go:go_term.name for (go, go_term) in basic_obo.items()}
#go_exclude = [go for (go, go_term) in basic_obo.items() if (len(go_term.children) > 50) or (go_term.level == 0)]

## Run GOATools

In [ ]:
GOEA = {}
GOEA_RES_SIG = {}

for t in terms:
    GOEA[t] = {}
    GOEA_RES_SIG[t] = {}
    dat = DESEQ[t]
    # drop Y'
    dat = dat.loc[~dat['gene'].isin(['Y_prime_short', 'Y_prime_long'])]
    ref = list(dat['gene'].map(sysname_to_sgdid))
    
    f_pval = dat['padj'] < 0.05
    f_incr = dat['log2FoldChange'] > 0
    f_decr = dat['log2FoldChange'] < 0
    
    for f, direction in zip([f_incr, f_decr], ['incr', 'decr']):
        qry = list(dat.loc[f_pval & f, 'gene'].map(sysname_to_sgdid))
        goea = GOEnrichmentStudyNS(ref, sgd_gaf.get_ns2assc(), basic_obo, methods=['fdr_bh'])
        goea_res_all = goea.run_study(qry)
        goea_res_sig = [r for r in goea_res_all if r.p_fdr_bh < 0.05]
        GOEA_RES_SIG[t][direction] = goea_res_sig
        tsv_path = f'{base_dir}goatools/goea.{t}.{direction}.tsv'
        goea.wr_tsv(tsv_path, goea_res_sig)
        goea = pd.read_csv(tsv_path, sep='\t')
        goea['direction'] = direction
        GOEA[t][direction] = goea

In [ ]:
for t in terms:
    direction = 'decr'
    plot_results(f'{fig_path}{t}_{direction}'+"_{NS}.png", GOEA_RES_SIG[t][direction])

# HU contrast

In [ ]:
# Import protein l2fc in HU from ncb 2012
prot_l2fc = pd.read_csv(f'{base_dir}goatools/tkach_ncb2012_S1.csv')
prot_l2fc['sgdid'] = prot_l2fc['Systematic ORF'].map(sysname_to_sgdid)

# Y' contrast

### Test for the enrichment of regulators

In [ ]:
REG = {}
for t in ['single_yprime.y_prime_kb']:
    
    reg = DESEQ[t].copy()
        
    reg = reg.loc[~reg['gene'].isin(['Y_prime_short', 'Y_prime_long', 'yprime'])].set_index('gene')
    
    G_all = reg.index
    G_sign = reg.loc[reg['padj']<0.05].index
    reg['genes_all'] = len(G_all)
    reg['genes_sign'] = len(G_sign)
    
    idx = 0
    with ProgressBar(max_value=len(G_all)) as bar:
        for q in G_all:
            r_all = sum([q in Regulation[s] for s in G_all])
            r_sign = sum([q in Regulation[s] for s in G_sign])
            
            reg.loc[q, 'reg_all'] = r_all
            reg.loc[q, 'reg_sign'] = r_sign
    
            idx += 1
            bar.update(idx)

    for i, (r_all, g_all, r_sign, g_sign) in reg[['reg_all', 'genes_all', 'reg_sign', 'genes_sign']].iterrows():
        ct = np.array([[r_sign, r_all-r_sign], [g_sign, g_all-g_sign]])
        fe = stats.fisher_exact(ct)
        reg.loc[i, 'odds'] = fe.statistic
        reg.loc[i, 'fe_pvalue'] = fe.pvalue
        if not np.any(np.array([r_all, g_all, r_sign, g_sign])==0):
            reg.loc[i, 'fold_enrich'] = (r_sign/r_all)/(g_sign/g_all)

    reg['fe_padj'] = multipletests(reg['fe_pvalue'], method='fdr_bh')[1]
    reg['log_fe_padj'] = -1*np.log10(reg['fe_padj'])
    
    REG[t] = reg

## Fig 4AB

In [ ]:
fig = plt.figure(figsize=[10, 3])

gs = plt.GridSpec(ncols=2, nrows=2, wspace=0.3, hspace=0.05, width_ratios=[4,3], height_ratios=[1,3],
                 left=0.08, right=0.8, top=0.85, bottom=0.15)

## Volcano plot

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0])

t = 'single_yprime.y_prime_kb'
reg = REG[t]
deseq = DESEQ[t].copy()
deseq['sign'] = deseq['padj'] < 0.05
deseq['direction'] = np.where(deseq['log2FoldChange']>0, 'up', 'down')
deseq['sign_reg'] = deseq['gene'].isin(reg.loc[reg['fe_padj']<0.05].index)
deseq['rap1_tar'] = ['YNL216W' in Regulation[s] if s != 'yprime' else False for s in deseq['gene'].values]
deseq['gcn4_tar'] = ['YEL009C' in Regulation[s] if s != 'yprime' else False for s in deseq['gene'].values]
gcn4_direction = regulation.loc[(regulation['Gene.regulatoryRegions.regulator.symbol']=='GCN4')]\
    .groupby('Gene.secondaryIdentifier')\
    .apply(lambda x: x['Gene.regulatoryRegions.regulationDirection'].unique()[0], include_groups=False)\
    .fillna('unknown').rename('gcn4_dir')
deseq = deseq.merge(gcn4_direction, left_on='gene', right_index=True, how='left')

for i, (g, sign_reg, rap1_tar, gcn4_tar, gcn4_dir) in deseq[['gene', 'sign_reg', 'rap1_tar', 'gcn4_tar', 'gcn4_dir']].iterrows():
    
    if sign_reg:
        annot = 'enrich. reg'
        z = 3
    
    elif gcn4_tar:
        if gcn4_dir == 'unknown':
            annot = 'GCN4 target'
            z = 1
        elif gcn4_dir == 'positive':
            annot = 'GCN4 (+) target'
            z = 2
        elif gcn4_dir == 'negative':
            annot = 'GCN4 (\N{MINUS SIGN}) target'
            z = 2
    
    elif g == 'yprime':
        z = 2
        annot = 'Y\' element'
    
    else:
        annot = False
        z = 0
    
    deseq.loc[i, 'annot'] = annot
    deseq.loc[i, 'zorder'] = z

annot_color = {'Y\' element': 'orange',
               'enrich. reg': 'k', 
               'GCN4 target': 'mediumpurple', 
              'GCN4 (\N{MINUS SIGN}) target': 'royalblue', 
              'GCN4 (+) target': 'indianred'}


ax2.scatter(deseq['log2FoldChange'], deseq['log_padj'], c='0.8', s=4)
 
for ax, plot_legend in zip((ax1, ax2), (False, True)):
    
    sns.scatterplot(x='log2FoldChange', y='log_padj', 
                    hue='annot', palette=annot_color, s=20,
                    hue_order=annot_color.keys(),
                    data=deseq.loc[deseq['annot']!=False].sort_values(by='zorder'),
                   ax=ax, legend=plot_legend)
    ax.set_xlim(-6,6)


texts = ['YEL009C', 'YNL216W']
f_texts = deseq['gene'].isin(texts)

x = deseq.loc[f_texts, 'log2FoldChange'].values
y = deseq.loc[f_texts, 'log_padj'].values

ta.allocate(ax, x, y, sysname_to_name.loc[texts, 'name'],
            textcolor='k', linecolor='k', margin=0.005,
            linewidth=0.5, 
            min_distance=0.05, direction='northeast',
            avoid_label_lines_overlap=True, avoid_crossing_label_lines=True)

ax1.set_ylim(117, 133)
ax1.set_yticks([120, 130])
ax1.set_xticks([])
ax1.set_xlabel('')
ax1.set_ylabel('')
ax2.set_ylim(-1, 46)
ax2.set_xlabel('')
ax2.set_ylabel('')
for sp in ['top', 'right', 'bottom']:
    ax1.spines[sp].set_visible(False)
for sp in ['top', 'right']:
    ax2.spines[sp].set_visible(False)

ax2.axhline(-1*np.log10(0.05), ls='--', lw=1, c='k', zorder=4)


kwargs = dict(marker=[(-1, -0.5), (1, 0.5)], markersize=8,
              linestyle="none", color='k', mec='k', mew=1, clip_on=False)
ax1.plot([0], [0], transform=ax1.transAxes, **kwargs)
ax2.plot([0], [1], transform=ax2.transAxes, **kwargs)

ax2.legend(loc=8, bbox_to_anchor=[0.5, 0.85], fancybox=False)


ax2.set_xlabel('L2FC mRNA')
ax2.text(-0.18, 0.67, '-log$_{10}$ FDR p-value', 
         va='center', rotation=90, transform=ax2.transAxes)

# Cumulative distribution of transcript levels

ax = fig.add_subplot(gs[:, 1])

rpm = RPM.loc[RPM['IC']==0].groupby(['background','gene'])['rpm'].mean()

legend_elms = []

for s in ['YJM967', 'YJM964', 'YJM947', 'YJM955']:
    dat = rpm.loc[s].copy()
    dat = dat.sort_values()
    q = dat.rank().loc['yprime']/dat.shape[0]*100

    c = strain_yprime_color[s]
    
    ax.plot(dat, np.linspace(0, 100, dat.shape[0]), color=c)
    ax.axvline(dat.loc['yprime'], c=strain_yprime_color[s], lw=1, ls='--')
    ax.axhline(q, c=strain_yprime_color[s], lw=1, ls='--')
    
    legend_elms.append(Line2D([0], [0], lw=1, color=c, label=f'{s}: {q:.3f}'))


ax.legend(handles=legend_elms, title='Y\' RPM\npercentile',
          loc=6, bbox_to_anchor=[1, 0.5], frameon=False)

ax.set_ylim(50, 105)
ax.set_xlim(10, ax.get_xlim()[1])

ax.set_xscale('log')
ax.set_xlabel('RPM')
ax.set_ylabel('% genes')

sns.despine(ax=ax)

fig.text(0.01, 0.89, 'A', size=24, fontweight='bold')
fig.text(0.47, 0.89, 'B', size=24, fontweight='bold')


for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig4AB.{ext}', dpi=300)

plt.show()
plt.close()

## Fig 4CD

In [ ]:
fig = plt.figure(figsize=[10, 5])

#gs = plt.GridSpec(ncols=4, nrows=1, width_ratios=[2,1,0.5,2],  wspace=0.3,
#                 left=0.35, right=0.98, top=0.76, bottom=0.09)
gs = plt.GridSpec(ncols=2, nrows=1, wspace=0.8,
                 left=0.26, right=0.97, top=0.9, bottom=0.09)

greys_odds = LinearSegmentedColormap.from_list('odds', list(zip([0,1], ['#dddddd', '#000000'])))
color_direction = {1:cm.coolwarm(1.0), -1:cm.coolwarm(0.0)}

## DESEQ GO enrich

t = 'single_yprime.y_prime_kb'
deseq = DESEQ[t].set_index('gene')

goea = pd.concat(GOEA[t].values()).replace({'direction':{'incr':1, 'decr':-1}})

goea = goea.loc[goea['enrichment']=='e'].sort_values(by='p_fdr_bh')\
    .groupby('direction').apply(lambda x: x.iloc[:20], include_groups=False).reset_index()
goea['log_padj'] = (-1)*np.log10(goea['p_fdr_bh'])
goea['study_items'] = goea['study_items'].apply(lambda x: np.array(x.split(', ')))
goea_expl = goea.explode('study_items')
goea_expl['gene'] = goea_expl['study_items'].map(sgdid_to_sysname)
goea_expl['l2fc'] = deseq.loc[goea_expl['gene'], 'log2FoldChange'].values
goea['avg_l2fc'] = goea_expl.groupby('name')['l2fc'].mean().sort_values().loc[goea['name']].values
goea['avg_l2fc_dir'] = goea['avg_l2fc']>0

goea_sub = goea.sort_values(by=['avg_l2fc_dir', 'log_padj'])
G = goea_sub['name'].values
Y = np.arange(len(G))
goea_sub['rank'] = Y

def rename_go_term(x):
    y = x.replace('biosynthetic', 'biosyn.')\
        .replace('organelle', 'orgl.')\
        .replace('process', 'proc.')\
        .replace('amino acid', 'a.a.')
    return y
    
y_labels = goea_sub.apply(lambda x: f'{rename_go_term(x["name"])} ({x["NS"]})', axis=1)
y_labels_colors = dict(zip(y_labels, [cm.coolwarm(i) for i in np.repeat([0.0, 1.0], 20)]))

ax1 = fig.add_subplot(gs[0])

for d, df in goea_sub.groupby('direction'):
    ax1.barh(df['rank'], df['avg_l2fc'], color=color_direction[d], alpha=0.3)

ax = ax1.twiny()

sns.scatterplot(y='rank', x='log_padj', 
                size='study_count', sizes=(10, 200),
                hue='fold_enrichment', palette=greys_odds,
                data=goea_sub, ax=ax, clip_on=False)

ax1.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
ax1.set_xlabel('average L2FC mRNA')
ax1.xaxis.set_label_position('top')

ax.tick_params(top=False, labeltop=False, bottom=True, labelbottom=True)
ax.set_xlabel('-log$_{10}$ FDR p-value')
ax.xaxis.set_label_position('bottom')

ax.legend(loc=3, ncol=1, bbox_to_anchor=[1.05, 0.05], 
          handlelength=1, fancybox=False)

leg = ax.legend_
for tx in leg.get_texts():
    if tx.get_text() == 'study_count':
        tx.set_text('# DE\ngenes')
    elif tx.get_text() == 'fold_enrichment':
        tx.set_text('fold\nenrich.')

ax.set_ymargin(0.02)
ax1.set_ymargin(0.02)
ax1.set_ylabel('')
ax1.set_yticks(Y)
ax1.set_yticklabels(y_labels, size=8)

for yt in ax1.get_yticklabels():
    ytx = yt.get_text()
    c = y_labels_colors[ytx]
    yt.set_color(c)


## Regulators

dat = REG[t].copy().reset_index()
dat['rank'] = dat['fe_padj'].rank(method='first')
dat['sign_reg'] = (dat['fe_padj']<0.05).replace({True:'regulator', False:'other'})
dat['rap1_reg'] = dat['gene'].isin(targets['YNL216W']).map({True:'target', False:'other'})
dat['gcn4_reg'] = dat['gene'].isin(targets['YEL009C'])

gcn4_direction = regulation.loc[(regulation['Gene.regulatoryRegions.regulator.symbol']=='GCN4')]\
    .groupby('Gene.secondaryIdentifier')\
    .apply(lambda x: x['Gene.regulatoryRegions.regulationDirection'].unique()[0], include_groups=False)\
    .replace({'positive':0, 'negative':1}).fillna(-1).rename('gcn4_dir')
dat = dat.merge(gcn4_direction, left_on='gene', right_index=True, how='left')
dat['Gcn4_dir'] = dat['gcn4_dir'].replace({-1:'no annot', 1:'negative', 0:'positive'})

palette_reg = {'regulator':'k', 'other':'0.86'}
palette_rap1 = {'target':'k', 'other':'0.86'}
palette_dir = dict(zip(['negative', 'no annot', 'positive'], [cm.coolwarm(i) for i in [0.0, 0.5, 1.0]]))

## Scatterplot of regulators

ax = fig.add_subplot(gs[1])

dat1 = dat.loc[dat['fe_padj']<0.05]
sns.scatterplot(x='log_fe_padj', y='rank', 
                size='reg_sign', sizes=(10, 200),
                hue='fold_enrich', palette=greys_odds,
                data=dat1, ax=ax)
Y = np.arange(dat1.shape[0])+1
ax.set_yticks(Y)
ax.set_yticklabels(sysname_to_name.loc[dat.set_index('rank').loc[Y, 'gene'], 'name'])
for yt in ax.get_yticklabels():
    if yt.get_text() in ['RAP1', 'GCN4']:
        yt.set_fontweight('bold')


ax.legend(loc=4, ncol=1, bbox_to_anchor=[1.05, 0.05], 
          handlelength=1, fancybox=False)
    
leg = ax.legend_
for tx in leg.get_texts():
    if tx.get_text() == 'reg_sign':
        tx.set_text('# DE\ntargets')
    elif tx.get_text() == 'fold_enrich':
        tx.set_text('fold\nenrich.')


ax.set_xlabel('-log$_{10}$ FDR p-value')

ax.set_ymargin(0.02)
ax.set_ylabel('')
ax.invert_yaxis()

fig.text(0.01, 0.93, 'C', size=24, fontweight='bold')
fig.text(0.65, 0.93, 'D', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig4CD.{ext}', dpi=300)

plt.show()
plt.close()

## Fig S8A

In [ ]:
dat = RPM.pivot_table(index='gene', columns='sample', values='rpm')
dat = dat.loc[(dat==0).sum(axis=1)==0]
dat = np.log10(dat)

S = dat.columns
col_colors = pd.concat([strains_info.loc[S, 'background'].map(strain_yprime_color), 
                        strains_info.loc[S, 'IC'].apply(lambda x: cm.Reds(x/75))], axis=1)

clm = sns.clustermap(dat, robust=True, cmap='binary',
                     method='ward', cbar_kws={'label':'$log_{10}$ RPM'},
                     dendrogram_ratio=(0.1, 0.15), cbar_pos=(0.02, 0.83, 0.015, 0.1),
                    col_colors=col_colors, figsize=[11,7])
ax = clm.ax_heatmap
ax.set_xticks(np.arange(dat.shape[1])+0.5)
ax.set_xticklabels([s.replace('_', ' ') for s in S[clm.dendrogram_col.reordered_ind]], rotation=90)
ax.set_xlabel('')
ax.set_yticks([])
ax.set_ylabel('')

clm.fig.text(0.01, 0.99, 'A',  size=24, fontweight='bold', ha='left', va='top')

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS8A.{ext}', dpi=300)
    
plt.show()
plt.close()

## Fig S8B-D

In [ ]:
fig = plt.figure(figsize=[11, 3])
gs = plt.GridSpec(ncols=4, nrows=1, wspace=0.45,
                 left=0.05, top=0.86, right=0.97, bottom=0.16)


# transcriptome signature HU

ax = fig.add_subplot(gs[0])

deseq = DESEQ['single_IC.IC']
l2fc = deseq.merge(prot_l2fc, left_on='gene', right_on='Systematic ORF', how='left')
l2fc['direction'] = np.where(l2fc['log2FoldChange']>0, 'incr', 'decr')

dat = l2fc.loc[(l2fc['padj']<0.05) & (~l2fc['HU ratio log2'].isna())]
u,v = dat.groupby('direction')['HU ratio log2']
ttest = stats.ttest_ind(u[1], v[1])

sns.boxplot(x='direction', y='HU ratio log2', 
            order=['decr','incr'], 
            color='w', linecolor='k',
            data=dat, ax=ax, zorder=1)

X = [0,1]
y = 3.3
ax.plot(X, [y, y], c='k', lw=2)
ax.text(np.mean(X), y+0.1, f'p={mh.plot_pval_text(ttest.pvalue)}', 
        ha='center', c='k')

# Annotate top hit
g = 'YML058W-A'
y = dat.set_index('gene').loc[g, 'HU ratio log2']
x = 1
ax.plot([x, x-0.4], [y, y-0.4], color='r', zorder=0)
ax.text(x-0.45, y-0.45, sysname_to_name.loc[g, 'name'], ha='right', va='top', color='r')

ax.set_xticks(X, labels=['< 0', '> 0'])
ax.set_xlabel('L2FC mRNA')

ax.set_ylabel('L2FC GFP')

ax.text(-0.25, 1.06, 'B', size=24, fontweight='bold', transform=ax.transAxes)


# RPM Yprime

ax = fig.add_subplot(gs[1])

dat = RPM.loc[RPM['gene']=='yprime'].copy()
dat['rpm_per_cn'] = dat['rpm']/dat['y_prime_cn']

sns.lineplot(x='IC', y='rpm', 
             hue='background', palette=strain_yprime_color, 
             hue_order=['YJM967','YJM964','YJM947','YJM955'],
             units='rep', estimator=None,
             data=dat, ax=ax, legend=False)

ax.set_yscale('log')
ax.set_ylabel('RPM')
ax.set_xticks([0, 25, 50, 75])
ax.set_xticklabels(['SC', 'IC25', 'IC50', 'IC75'])
ax.set_xlabel('HU')


ax.text(-0.25, 1.06, 'C', size=24, fontweight='bold', transform=ax.transAxes)

ax = fig.add_subplot(gs[2])

sns.lineplot(x='IC', y='rpm_per_cn', 
             hue='background', palette=strain_yprime_color, 
             hue_order=['YJM967','YJM964','YJM947','YJM955'],
             units='rep', estimator=None,
             data=dat, ax=ax, legend=False)

ax.set_ylabel('RPM per copy')
ax.set_xticks([0, 25, 50, 75])
ax.set_xticklabels(['SC', 'IC25', 'IC50', 'IC75'])
ax.set_xlabel('HU')

ax = fig.add_subplot(gs[1:3])
ax.axis('off')
legend_elms = [Line2D([0], [0], color=strain_yprime_color[s], label=s) for s in ['YJM967','YJM964','YJM947','YJM955']]
ax.legend(handles=legend_elms, ncol=4, loc=9, columnspacing=1.5, bbox_to_anchor=[0.5, 1.18], frameon=False)


# deseq yprime

ax = fig.add_subplot(gs[3])

term_symbol = {'complete.y_prime_kb':'o', 'complete.IC':'s', 'complete.IC.y_prime_kb':'X'}
term_alias = {'complete.y_prime_kb':'Y\' kb', 'complete.IC':'HU', 'complete.IC.y_prime_kb':'Y\' kb '+r'$\times$ HU'}

dat = pd.concat(DESEQ.values()).set_index(['gene','term']).loc['yprime'].loc[['complete.IC', 'complete.y_prime_kb', 'complete.IC.y_prime_kb']].reset_index()
sns.scatterplot(x='log2FoldChange', y='log_padj', c='k', s=72,
                style='term', markers=term_symbol,
                data=dat)

ax.set_xlabel('L2FC Y\' mRNA')
ax.set_ylabel('-log$_{10}$ FDR p-value')
ax.axhline(-1*np.log10(0.05), ls='--', lw=1, c='k', zorder=0)

legend_elms = [Line2D([0], [0], c='w', marker=m, ms=8, mfc='k', mew=0, label=term_alias[t]) for t,m in term_symbol.items()]

ax.legend(handles=legend_elms, loc=2, bbox_to_anchor=[0, 1], fancybox=False)

ax.text(-0.25, 1.06, 'D', size=24, fontweight='bold', transform=ax.transAxes)

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS8B-D.{ext}', dpi=300)

plt.show()
plt.close()

# Correlation between single and interaction contrase

In [ ]:
DESEQ_COMP = []
t0 = 'IC.y_prime_cn'
for t1 in ['single.y_prime_cn', 'y_prime_cn']:
    deseq_comp = []
    for t, t_a in zip([t0, t1], ['inter', 'single']):
        deseq = DESEQ[t].set_index('gene')
        deseq = deseq.rename({c:f'{t_a}.{c}' for c in deseq.columns}, axis=1)
        deseq_comp.append(deseq)
    deseq_comp = pd.concat(deseq_comp, axis=1)
    for g, (p1, p2) in (deseq_comp[['inter.padj', 'single.padj']]<0.05).iterrows():
        if p1 and p2:
            sign = 'both'
        elif p1 and not p2:
            sign = 'single'
        elif p2 and not p1:
            sign = 'inter'
        else:
            sign = 'none'
        deseq_comp.loc[g, 'significance'] = sign
    deseq_comp['contrast'] = t1
    DESEQ_COMP.append(deseq_comp)
    
DESEQ_COMP = pd.concat(DESEQ_COMP).reset_index(drop=True)

In [ ]:
fig = plt.figure(figsize=[12,5])
gs = plt.GridSpec(ncols=3, nrows=1, wspace=0.3,
                 left=0.06, top=0.80, right=0.97, bottom=0.12)

term_ax = {'single.y_prime_cn':1, 'y_prime_cn':2}
term_alias = {'single.y_prime_cn':'single contrast', 'y_prime_cn':'full model'}
sign_alias = {'none':'none', 'single':'~ Y\' CN', 'inter':'~ Y\' CN '+r'$\times$ HU', 'both':'both'}
sign_color = {'none':'0.75', 'single':'dodgerblue', 'inter':'indianred', 'both':'indigo'}

ax = fig.add_subplot(gs[0])

g = 'YEL009C'
gn = sysname_to_name.loc[g, 'name']
dat = RPM.loc[RPM['gene']==g].copy()
dat['log_rpm'] = np.log10(dat['rpm'])

sns.lineplot(x='IC', y='log_rpm', 
             hue='background', palette=strain_yprime_color, 
             hue_order=['YJM967','YJM964','YJM947','YJM955'],
             units='rep', estimator=None,
             data=dat, ax=ax)

ax.legend(loc=8, ncol=2, bbox_to_anchor=[0.5,1.05], title='', frameon=False)
ax.set_ylabel(gn+' $log_{10}$ RPM')
ax.set_xticks([0, 25, 50, 75])
ax.set_xticklabels(['SC', 25, 50, 75])
ax.set_xlabel('HU IC')
ax.text(-0.17, 1.04, 'A', size=24, fontweight='bold', transform=ax.transAxes)


for t, df in DESEQ_COMP.groupby('contrast'):
    
    df = df.copy()
    rank = df.value_counts('significance').rank(ascending=False)
    df['rank'] = rank.loc[df['significance']].values
    df = df.sort_values(by='rank')

    ax_idx = term_ax[t]
    ax = fig.add_subplot(gs[ax_idx])
    
    sns.scatterplot(x='single.log2FoldChange', y='inter.log2FoldChange', 
                   hue='significance', palette=sign_color,
                   data=df, ax=ax, legend=False)
    
    ax.set_xlabel('L2FC ~ Y\' CN')
    ax.set_ylabel('L2FC ~ Y\' CN '+r'$\times$ HU')
    ax.text(0.5, 1, term_alias[t], size=12, ha='center', va='top', transform=ax.transAxes)

    ax.text(-0.17, 1.04, 'ABC'[ax_idx], size=24, fontweight='bold', transform=ax.transAxes)

ax = fig.add_subplot(gs[1:])
ax.axis('off')
legend_elms = [Line2D([0], [0], c='w', marker='o', ms=8, mfc=sign_color[s], label=sign_alias[s]) for s in ['none', 'single', 'inter', 'both']]
ax.legend(handles=legend_elms, loc=8, ncol=4, bbox_to_anchor=[0.5, 1.13], 
          frameon=False, title='significance')

sns.despine()
    
#plt.show()
plt.close()

In [ ]:
deseq = DESEQ['single.y_prime_cn']

#for direction, goea in GOEA['single.y_prime_cn'].items():  
    #for ns in ['BP', 'MF', 'CC']:

direction = 'decr'
goea = GOEA['single.y_prime_cn'][direction]
goea = pd.concat(GOEA['single.y_prime_cn'].values())
ns = 'BP'
        
goea_sub = goea.loc[(goea['enrichment']=='e') 
    & (goea['NS']==ns)
    & (~goea['# GO'].isin(go_exclude))
    ].set_index('# GO', drop=False).copy()

if goea_sub.shape[0] > 0:

    goea_sub['sgdid_list'] = goea_sub['study_items'].apply(lambda x: np.array(x.split(', ')))
    
    S = np.unique(np.concatenate(goea_sub['sgdid_list'].values))
    S = [sgdid_to_sysname[s] for s in S]

    S = deseq.set_index('gene').loc[S].sort_values(by='log2FoldChange', ascending=True).index
    
    go_annot_matrix = pd.DataFrame(np.zeros((len(S), goea_sub.shape[0])), index=S, columns=goea_sub.index)
    for go in go_annot_matrix.columns:
        go_annot_matrix[go] = np.where([sysname_to_sgdid[s] in goea_sub.loc[go, 'sgdid_list'] for s in S], 1, 0)

    
    figsize = np.flip(np.array(go_annot_matrix.shape)*0.1 + np.array([1,3]))

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    
    gs = plt.GridSpec(ncols=2, nrows=2, figure=fig,
                      height_ratios=[1,9], width_ratios=[9,1],
                     hspace=0.03, wspace=0.05)

    X = np.arange(go_annot_matrix.shape[1])
    Y = np.arange(go_annot_matrix.shape[0])

    ax = fig.add_subplot(gs[1,0])
    ax.imshow(go_annot_matrix, cmap='binary', interpolation='nearest', aspect='auto')
    
    ax.set_xticks(X)
    ax.set_xticklabels(goea_sub['name'], size=6, rotation=90)
    ax.set_yticks(Y)
    ax.set_yticklabels(sysname_to_name.loc[S, 'name'], size=5)
    ax.invert_yaxis()
    
    ax = fig.add_subplot(gs[0,0])
    ax.bar(X, -1*np.log10(goea_sub['p_fdr_bh']), width=1, color='k')
    ax.set_xlim(X[[0, -1]]+np.array([-0.5, 0.5]))
    ax.set_xticks([])
    ax.set_ylabel('$-log_{10}$\np-value', rotation=0, ha='right', va='center')

    ax = fig.add_subplot(gs[1,1])
    ax.barh(Y, deseq.set_index('gene').loc[S, 'log2FoldChange'], height=1, color='k')
    ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
    #ax.set_yticks([])
    ax.set_yticks(Y)
    ax.set_yticklabels(sysname_to_name.loc[S, 'name'], size=5)
    ax.set_xlabel('L2FC mRNA', rotation=90)
    ax.tick_params(axis='x', rotation=90)

    #plt.show()
    plt.close()

In [ ]:
telo_genes = pd.read_csv(f'{base_dir}script/telo_genes.txt', header=None).rename({0:'name'}, axis=1)
telo_genes = telo_genes.loc[telo_genes['name']!='TLC1']
telo_genes['sys_name'] = sysname_to_name.set_index('name').loc[telo_genes['name'], 'sys_name'].values

In [ ]:
t = 'single.y_prime_cn'
dat = DESEQ[t].merge(telo_genes, left_on='gene', right_on='sys_name')

cl = sns.clustermap(dat['log2FoldChange'], figsize=[3,6],
                    cbar_kws={'orientation':'horizontal'}, col_cluster=False, 
                    dendrogram_ratio=(0.4, 0.2), #cbar_pos=(0.2, 0.9, 0.4, 0.03),
                    cmap='PuOr', center=0, vmin=-2.5, vmax=2.5)
rdr_idx = cl.dendrogram_row.reordered_ind

ax = cl.ax_heatmap

ax.set_yticks(np.arange(dat.shape[0])+0.5)
ax.set_yticklabels(dat.iloc[rdr_idx]['name'].values, rotation=0)
ax.set_ylabel('')
ax.set_xticks([])
ax.set_xlabel('')

(x0,y0), (x1,y1) = np.array(cl.ax_heatmap.get_position())
ax = cl.ax_cbar
ax.set_position([x0, 0.9, x1-x0, 0.03])
ax.set_xlabel('L2FC mRNA')


plt.savefig(f'{fig_path}yprime_l2fc_telo_genes.png', dpi=300, bbox_inches="tight")
#plt.show()
plt.close()

In [ ]:
 # Labels for top hits
    texts = []
    for gi, (gn, x, y) in deseq.loc[top_hits, ['gene_name', 'log2FoldChange', 'log10_padj']].iterrows():

        if gn == 'ERG11':
            c = 'limegreen'
        else:
            c = 'k'
        texts.append(ax.text(x, y, gn, color=c, size=8, ha='center', va='center'))

    X = deseq.loc[top_hits, 'log2FoldChange']
    Y = deseq.loc[top_hits, 'log10_padj']
    adjust_text(texts, expand=(2, 3), x=X, y=Y, arrowprops=dict(arrowstyle="-", color='k', lw=0.5))

# Expression with genomic coordinate

In [ ]:
counts_um = pd.read_csv(f'{base_dir}tables/counts.um.csv', index_col=0)
counts_um.columns = [re.match(r'YJM\d{3}_(?:SC|IC25|IC50|IC75)_\d', c).group(0) for c in counts_um.columns]
# transform into rpm long-format table
RPM_UM = counts_um/counts_um.sum()*1e6
RPM_UM.loc['yprime'] = RPM_UM.loc['Y_prime_long'] + RPM_UM.loc['Y_prime_short']
RPM_UM = RPM_UM.reset_index(names='gene').melt(id_vars='gene', var_name='lib', value_name='rpm')
RPM_UM['sample'] = RPM_UM['lib'].apply(lambda x: x.split('.')[0])
RPM_UM = RPM_UM.merge(strains_info.reset_index())

In [ ]:
terms = ['single_yprime.um.y_prime_kb']

DESEQ_UM = {}
for t in terms:
    deseq = pd.read_csv(f'{base_dir}tables/deseq.{t}.csv').rename({'Unnamed: 0':'gene'}, axis=1)
    deseq = deseq.loc[~deseq['log2FoldChange'].isna()]
    deseq['log_padj'] = np.where(deseq['padj'].isna(), np.nan, np.log10(deseq['padj'])*(-1))
    deseq['term'] = t
    DESEQ_UM[t] = deseq

In [ ]:
dat = deseq.merge(gff_annot_exploded.rename({'gene':'gene_gff'}, axis=1), left_on='gene', right_on='ID')
dat['avg_pos'] = dat[[3,4]].mean(axis=1)
dat['chrom_len'] = tig_off.loc[dat[0], 'len'].values
dat['pos_abs'] = dat.apply(lambda x: min(x['avg_pos'], x['chrom_len']-x['avg_pos']), axis=1)

In [ ]:
fig, ax = plt.subplots()

for chrom, df in dat.loc[(dat['pos_abs']<5e4) & 
    (dat[0]!='ref|NC_001224|')].sort_values(by='pos_abs').groupby(0):

    color = colorcet.glasbey_bw[int(chrom[11:13])-33]
    ax.plot(df['pos_abs'], df['log2FoldChange'], color=color, label=chrom)
    print(color)

plt.legend(loc=2, bbox_to_anchor=[1,1])

plt.show()
plt.close()